# Samudra-style first-pass OHC emulator

**Goal.** Build a quick experimental emulator to test whether a neural emulator drifts away from ACCESS-OM2 truth over a 10-year rollout.

This is **not** a full Samudra reproduction. It is a first pass that keeps the existing `AutoEncoder_om2.ipynb` workflow where possible:

- use the same ACCESS-OM2 OHC / surface heat-flux data accessor;
- use the same PET pipeline / PET normalisation / PET undo workflow;
- use the existing `UNet`, `AutoEncoder`, `PartialConv2d`, `build_normalisation`, and `make_fast_dl` utilities from `src`;
- change the task from autoencoding/reconstruction to **one-step forecasting**.

## Learning task

The default task is Samudra-style autoregressive forecasting:

\[
\left[\mathrm{OHC}(t), \mathrm{HF}(t)\right] \rightarrow \mathrm{OHC}(t+1)
\]

where `HF` is `total_surface_heat_flx` and `OHC` is `ocean_heat_content_2d`.

A strict **HF-only** ablation is also included:

\[
\mathrm{HF}(t) \rightarrow \mathrm{OHC}(t+1)
\]

but that version is less physically stateful and is not a true drift test because the previous predicted OHC is not fed back into the next step.


## Paper-derived design choices

The notebook mirrors the emulator framing in the linked ocean-emulator papers:

1. Treat surface flux / atmospheric boundary information as a forcing.
2. Predict the next ocean state from the current ocean state plus forcing.
3. Roll the model forward autoregressively for several years.
4. Diagnose accumulated bias and drift, not only one-step skill.

For this first pass, the network is intentionally simple and reuses your existing UNet implementation. The main deliverable here is the **data plumbing + rollout diagnostic**, not the final emulator architecture.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: import dependencies and define paths/device for the first-pass
# Samudra-style OHC emulator.
#
# This follows the existing AutoEncoder_om2.ipynb setup as closely as possible.
# -----------------------------------------------------------------------------

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# PyEarthTools pipeline
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

# Visualisation
import matplotlib.pyplot as plt

# Reproducibility / performance
torch.manual_seed(42)
try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------------------------------------------------------
# Source data and code directories
# -----------------------------------------------------------------------------
userbase = "/g/data/nm47/txs156/"

# Taimoor's repobase:
repobase = "/home/156/txs156/uom/OM2-emulator/"

# Ryan's repobase:
# repobase = "/home/561/rmh561/ML/OM2-emulator/"

# Navid's repobase:
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

src_path = Path(repobase) / "src"
sys.path.insert(0, str(src_path))

from Data import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import AutoEncoder, LightningWrapper, PartialConv2d, UNet


In [ ]:
# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------

# Source file from the existing AutoEncoder notebook
datapath = userbase + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

norm_variables = ["ocean_heat_content_2d", "total_surface_heat_flx"]

# Normalisation / sampling
norm_strat = "Spatial_climatology"
time_interval = "1MS"

# First-pass split:
# - train on the earlier part of the data;
# - roll forward for ~10 years after the training window.
#
# DateRange appears to behave as [start, end), so the end dates below are
# written as the first month AFTER the desired final month.
time_start = "2000-01"
train_end_exclusive = "2008-12"

# Rollout needs 1 initial OHC month + 120 target months for a 10-year forecast.
rollout_initial_start = "2008-12"
rollout_end_exclusive = "2019-01"
rollout_years = 10
rollout_months_requested = 12 * rollout_years

# Training controls
batch_size = 16
source_batch_size = 32
max_epochs = 100
learning_rate = 1e-4
num_workers = 0

# Reserve the last N one-step samples from the training period as an
# in-distribution validation set.
validation_months_from_training = 24

In [ ]:
# -----------------------------------------------------------------------------
# Normalisation and ACCESS-OM2 accessor
# -----------------------------------------------------------------------------
#
# This is directly inherited from AutoEncoder_om2.ipynb.
# build_normalisation returns the land/ocean mask and the PET normalisation
# transform used inside the pipeline.
# -----------------------------------------------------------------------------

mask, normalisation = build_normalisation(
    datapath,
    norm_strat,
    norm_variables,
    time_window=dict(start=time_start, end=rollout_end_exclusive, freq=time_interval),
    train_end=train_end_exclusive,
    mask=True,
)

mean = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]

ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t"] + norm_variables,
    root=userbase + "OM2-emulator/data/",
)


In [ ]:
import pyearthtools
import pyearthtools.pipeline as petpipe
import importlib.metadata as md

print("pyearthtools path:", pyearthtools.__file__)

for pkg in ["pyearthtools", "pyearthtools-pipeline", "pyearthtools_pipeline"]:
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        pass

print("Has TemporalWindow?", hasattr(petpipe.modifications, "TemporalWindow"))
print("Available modifications:", [x for x in dir(petpipe.modifications) if "Temporal" in x])

In [ ]:
window_delta = time_end - time_start


In [ ]:
# -----------------------------------------------------------------------------
# PET pipeline builder
# -----------------------------------------------------------------------------
#
# This is the same pipeline as AutoEncoder_om2.ipynb, wrapped in a function so
# we can build separate pipelines for training, rollout inputs, and PET undo.
# -----------------------------------------------------------------------------
TRAIN_ROLLOUT_STEPS = 12   # debug first; 120 = 10 years monthly
TIME_DELTA_UNIT = "month"

iterator_interval = (1, TIME_DELTA_UNIT)
iterator_delta = petdata.TimeDelta(iterator_interval)
rollout_delta = petdata.TimeDelta((TRAIN_ROLLOUT_STEPS, TIME_DELTA_UNIT))

time_start = petdata.Petdt(time_start) + iterator_delta
time_end = petdata.Petdt(train_end_exclusive) - rollout_delta
window_delta = time_end - time_start

pipeline_i = petpipe.Pipeline(
        ACCESS_OHC_accessor,
        petdata.transforms.coordinates.Drop(["geolat_t", "geolon_t"]),
        petdata.transforms.variables.Drop(["area_t"]),
        petpipe.operations.xarray.select.SelectDataset(norm_variables),
        petpipe.operations.xarray.Sort(order=norm_variables, strict=True),
        petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
        normalisation,
        petdata.transforms.coordinates.Drop(["geolat_t", "geolon_t"]),
        petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
        petpipe.operations.xarray.conversion.ToNumpy(),
        petpipe.operations.numpy.reshape.Rearrange("c t h w -> t c h w"),
    # PET-native Samudra window:
        petpipe.modifications.TemporalWindow(
        prior_indexes=[-1, 0],
        posterior_indexes=list(range(0, TRAIN_ROLLOUT_STEPS + 1)),
        timedelta=window_delta,
        merge_method=concat_time_numpy),
        iterator=petpipe.iterators.DateRange(
        time_start,
        time_end,
        interval=iterator_interval))


In [ ]:
# -----------------------------------------------------------------------------
# Materialise PET data into memory once
# -----------------------------------------------------------------------------
#
# The original notebook found PET's default epoch-by-epoch loading too slow.
# We keep that optimisation: load PET -> PyTorch once using make_fast_dl().
# -----------------------------------------------------------------------------

splits = {
    "train_split": petpipe.iterators.DateRange(
        time_start, train_end_exclusive, interval="1 month"
    ),
    "valid_split": petpipe.iterators.DateRange(
        rollout_initial_start, rollout_end_exclusive, interval="1 month"
    ),
}

datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_i,
    **splits,
    batch_size=source_batch_size,
    num_workers=num_workers,
)

datamodule.setup("fit")

fast_train_source_dl = make_fast_dl(
    datamodule.train_dataloader(),
    batch_size=source_batch_size,
    shuffle=False,
    drop_last=False,
)

fast_rollout_source_dl = make_fast_dl(
    datamodule.val_dataloader(),
    batch_size=source_batch_size,
    shuffle=False,
    drop_last=False,
)


def batch_to_tensor(batch):
    """Robustly extract the tensor from PET/PyTorch batch formats."""
    if torch.is_tensor(batch):
        return batch

    if isinstance(batch, (tuple, list)):
        return batch[0]

    if isinstance(batch, dict):
        for key in ("x", "input", "inputs", "data"):
            if key in batch:
                return batch[key]

    raise TypeError(f"Cannot extract tensor from batch type: {type(batch)}")


def dataloader_to_tensor(dl):
    chunks = []
    for batch in dl:
        x = batch_to_tensor(batch)
        chunks.append(x.detach().cpu())
    return torch.cat(chunks, dim=0).float()


train_tensor = dataloader_to_tensor(fast_train_source_dl)
rollout_tensor = dataloader_to_tensor(fast_rollout_source_dl)

print("Materialised tensors:")
print(f"  train_tensor:   {tuple(train_tensor.shape)}  [time, channel, lat, lon]")
print(f"  rollout_tensor: {tuple(rollout_tensor.shape)} [time, channel, lat, lon]")

assert train_tensor.ndim == 4, "Expected train_tensor shape [time, channel, lat, lon]"
assert rollout_tensor.ndim == 4, "Expected rollout_tensor shape [time, channel, lat, lon]"
assert train_tensor.shape[1] == len(norm_variables), "Unexpected channel count"


In [ ]:
# -----------------------------------------------------------------------------
# Build one-step forecast pairs
# -----------------------------------------------------------------------------

def make_one_step_pairs(array, *, strict_hf_only=False):
    """
    Convert [time, channel, lat, lon] data into one-step forecast pairs.

    Default:
        X = [OHC(t), HF(t)]
        y = [OHC(t+1)]

    strict_hf_only=True:
        X = [HF(t)]
        y = [OHC(t+1)]
    """
    if torch.is_tensor(array):
        array = array.detach().cpu().numpy()

    ohc_t = array[:-1, OHC_CH:OHC_CH + 1, :, :]
    hf_t = array[:-1, HF_CH:HF_CH + 1, :, :]
    ohc_next = array[1:, OHC_CH:OHC_CH + 1, :, :]

    if strict_hf_only:
        X = hf_t
    else:
        X = np.concatenate([ohc_t, hf_t], axis=1)

    y = ohc_next
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


X_all, y_all = make_one_step_pairs(train_tensor, strict_hf_only=STRICT_HF_ONLY)

n_total = X_all.shape[0]
n_val = min(validation_months_from_training, max(1, n_total // 5))
n_train = n_total - n_val

X_train, y_train = X_all[:n_train], y_all[:n_train]
X_valid, y_valid = X_all[n_train:], y_all[n_train:]

train_dl = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=0,
)

valid_dl = DataLoader(
    TensorDataset(X_valid, y_valid),
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

print("One-step forecast tensors:")
print(f"  X_train: {tuple(X_train.shape)}")
print(f"  y_train: {tuple(y_train.shape)}")
print(f"  X_valid: {tuple(X_valid.shape)}")
print(f"  y_valid: {tuple(y_valid.shape)}")


In [ ]:
# -----------------------------------------------------------------------------
# Mask helpers
# -----------------------------------------------------------------------------

def extract_2d_mask(mask_like, *, channel=0):
    """Convert the PET/xarray mask into a 2D float mask [lat, lon]."""
    if hasattr(mask_like, "values"):
        arr = np.asarray(mask_like.values)
    else:
        arr = np.asarray(mask_like)

    # If mask includes variables/channels/time, take the first relevant slice.
    while arr.ndim > 2:
        arr = arr[channel if arr.shape[0] > channel else 0]

    return (arr > 0).astype(np.float32)


mask_2d = extract_2d_mask(mask, channel=OHC_CH)
print(f"mask_2d shape: {mask_2d.shape}; ocean fraction = {np.nanmean(mask_2d):.3f}")


In [ ]:
# -----------------------------------------------------------------------------
# Lightning wrapper for forward emulation
# -----------------------------------------------------------------------------
#
# We reuse the existing UNet model architecture, but use a new wrapper because
# the input and output channel counts differ:
#
#   Autoencoder wrapper: x -> x_hat, loss against same channels
#   Emulator wrapper:    x(t) -> OHC(t+1), loss against OHC only
# -----------------------------------------------------------------------------

class OHCForwardLightningWrapper(L.LightningModule):
    def __init__(self, model, mask_2d, lr=1e-4, weight_decay=0.0):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay

        mask_tensor = torch.as_tensor(mask_2d, dtype=torch.float32)[None, None, :, :]
        self.register_buffer("mask", mask_tensor)

    def forward(self, x):
        y_hat = self.model(x)
        return y_hat * self.mask

    def masked_mse(self, y_hat, y):
        valid = self.mask
        se = (y_hat - y) ** 2
        numerator = (se * valid).sum()
        denominator = valid.sum() * y.shape[0] * y.shape[1]
        return numerator / denominator.clamp_min(1.0)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.masked_mse(y_hat, y)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.masked_mse(y_hat, y)
        self.log("valid_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        return {"x": x.detach(), "y": y.detach(), "y_hat": y_hat.detach()}

    def configure_optimizers(self):
        return optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


In [ ]:
# -----------------------------------------------------------------------------
# Initialise the model
# -----------------------------------------------------------------------------

input_channel_count = 1 if STRICT_HF_ONLY else 2
output_channel_count = 1

base_model = UNet(
    input_channel_count=input_channel_count,
    output_channel_count=output_channel_count,
)

lightning_model = OHCForwardLightningWrapper(
    model=base_model,
    mask_2d=mask_2d,
    lr=learning_rate,
)

print(lightning_model)


In [ ]:
# -----------------------------------------------------------------------------
# Train the one-step emulator
# -----------------------------------------------------------------------------

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

trainer = L.Trainer(
    max_epochs=max_epochs,
    num_sanity_val_steps=0,
    accelerator=accelerator,
    devices=devices,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=True,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=train_dl,
    val_dataloaders=valid_dl,
)


In [ ]:
# -----------------------------------------------------------------------------
# One-step validation diagnostics
# -----------------------------------------------------------------------------

valid_preds = trainer.predict(lightning_model, dataloaders=valid_dl)

x_valid_pred = torch.cat([p["x"] for p in valid_preds], dim=0).cpu()
y_valid_true = torch.cat([p["y"] for p in valid_preds], dim=0).cpu()
y_valid_hat = torch.cat([p["y_hat"] for p in valid_preds], dim=0).cpu()

mask_t = torch.as_tensor(mask_2d)[None, None]
valid_mse = (((y_valid_hat - y_valid_true) ** 2) * mask_t).sum()
valid_mse = valid_mse / (mask_t.sum() * y_valid_true.shape[0] * y_valid_true.shape[1])
valid_rmse = torch.sqrt(valid_mse).item()

print(f"Normalised one-step validation RMSE: {valid_rmse:.4f}")


In [ ]:
# -----------------------------------------------------------------------------
# Autoregressive rollout
# -----------------------------------------------------------------------------
#
# Starting from the first OHC field in rollout_tensor, use the known HF sequence
# to predict OHC forward for up to 10 years.
# -----------------------------------------------------------------------------

@torch.no_grad()
def rollout_ohc(
    model,
    rollout_array,
    *,
    strict_hf_only=False,
    requested_months=120,
    device=None,
):
    """
    Autoregressively roll OHC forward.

    rollout_array:
        Tensor/array with shape [time, channel, lat, lon].
        The first time index supplies the initial OHC state.
        Subsequent target months are compared against truth.

    Returns
    -------
    pred_ohc : torch.Tensor
        Normalised predicted OHC at target months, shape [months, 1, lat, lon].
    """
    if device is None:
        device = next(model.parameters()).device

    if not torch.is_tensor(rollout_array):
        rollout_array = torch.as_tensor(rollout_array).float()

    rollout_array = rollout_array.float()
    available_months = rollout_array.shape[0] - 1
    rollout_months = min(requested_months, available_months)

    if rollout_months < requested_months:
        warnings.warn(
            f"Only {rollout_months} rollout months available; "
            f"requested {requested_months}."
        )

    model.eval()
    model.to(device)

    ohc_t = rollout_array[0:1, OHC_CH:OHC_CH + 1].to(device)
    preds = []

    for n in range(rollout_months):
        # Use forcing at the current step.
        hf_t = rollout_array[n:n + 1, HF_CH:HF_CH + 1].to(device)

        if strict_hf_only:
            x_t = hf_t
        else:
            x_t = torch.cat([ohc_t, hf_t], dim=1)

        ohc_next = model(x_t)
        preds.append(ohc_next.detach().cpu())

        # Feed the prediction back in. This is where drift accumulates.
        ohc_t = ohc_next.detach()

    return torch.cat(preds, dim=0)


rollout_pred_ohc_norm = rollout_ohc(
    lightning_model,
    rollout_tensor,
    strict_hf_only=STRICT_HF_ONLY,
    requested_months=rollout_months_requested,
    device=device,
)

rollout_months = rollout_pred_ohc_norm.shape[0]

# Truth/template target months are one month after the initial condition.
rollout_truth_norm = rollout_tensor[1:rollout_months + 1].cpu().numpy()
rollout_pred_norm = rollout_truth_norm.copy()
rollout_pred_norm[:, OHC_CH:OHC_CH + 1, :, :] = rollout_pred_ohc_norm.cpu().numpy()

print(f"Rolled out for {rollout_months} months ({rollout_months / 12:.2f} years)")
print(f"  truth/template array: {rollout_truth_norm.shape}")
print(f"  prediction array:     {rollout_pred_norm.shape}")


In [ ]:
# -----------------------------------------------------------------------------
# Undo PET normalisation for the rollout target months
# -----------------------------------------------------------------------------

def add_months(period_str, n_months):
    return str(pd.Period(period_str, freq="M") + n_months)

target_start = add_months(rollout_initial_start, 1)
target_end_exclusive = add_months(target_start, rollout_months)

print(f"Target months: {target_start} -> {target_end_exclusive} (exclusive)")

target_pipeline = make_ohc_hf_pipeline(target_start, target_end_exclusive)

truth_ds = target_pipeline.undo(rollout_truth_norm)
pred_ds = target_pipeline.undo(rollout_pred_norm)

truth_ohc = truth_ds[OHC_VAR]
pred_ohc = pred_ds[OHC_VAR]
error_ohc = pred_ohc - truth_ohc

print(truth_ohc)


In [ ]:
# -----------------------------------------------------------------------------
# Drift metrics
# -----------------------------------------------------------------------------
#
# We compute:
#   1. Area/cosine-weighted mean OHC time series.
#   2. Area/cosine-weighted mean OHC bias.
#   3. Area/cosine-weighted RMSE over time.
#
# If the PET undo has latitude/longitude coordinates, use cosine(latitude)
# weighting times the ocean mask. If not, fall back to mask-only weighting.
# -----------------------------------------------------------------------------

def infer_lat_lon_dims(da):
    dims = list(da.dims)
    lat_candidates = [d for d in dims if "lat" in d.lower() or "yt" in d.lower()]
    lon_candidates = [d for d in dims if "lon" in d.lower() or "xt" in d.lower()]

    if lat_candidates and lon_candidates:
        return lat_candidates[-1], lon_candidates[-1]

    # Fallback: assume final two dimensions are horizontal.
    return dims[-2], dims[-1]


lat_dim, lon_dim = infer_lat_lon_dims(truth_ohc)
print(f"Horizontal dims inferred as: {lat_dim}, {lon_dim}")

mask_da = xr.DataArray(
    mask_2d,
    dims=(lat_dim, lon_dim),
    coords={
        lat_dim: truth_ohc[lat_dim],
        lon_dim: truth_ohc[lon_dim],
    },
)

if lat_dim in truth_ohc.coords:
    try:
        lat_values = truth_ohc[lat_dim]
        lat_weights = xr.DataArray(
            np.cos(np.deg2rad(lat_values)),
            dims=(lat_dim,),
            coords={lat_dim: lat_values},
        )
        weights = lat_weights * mask_da
    except Exception as exc:
        warnings.warn(f"Could not construct cosine latitude weights: {exc}")
        weights = mask_da
else:
    weights = mask_da


def weighted_mean_ts(da, weights):
    return da.weighted(weights).mean(dim=(lat_dim, lon_dim))


def weighted_rmse_ts(error, weights):
    return np.sqrt((error ** 2).weighted(weights).mean(dim=(lat_dim, lon_dim)))


truth_mean = weighted_mean_ts(truth_ohc, weights)
pred_mean = weighted_mean_ts(pred_ohc, weights)
bias_mean = weighted_mean_ts(error_ohc, weights)
rmse_ts = weighted_rmse_ts(error_ohc, weights)

drift_summary = xr.Dataset(
    {
        "truth_mean_ohc": truth_mean,
        "pred_mean_ohc": pred_mean,
        "mean_bias_ohc": bias_mean,
        "spatial_rmse_ohc": rmse_ts,
    }
)

drift_summary


In [ ]:
# -----------------------------------------------------------------------------
# Plot 10-year drift diagnostics
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 9),
    constrained_layout=True,
)

truth_mean.plot(ax=axes[0], label="ACCESS-OM2 truth")
pred_mean.plot(ax=axes[0], label="Emulator rollout")
axes[0].set_title("Area-weighted mean OHC")
axes[0].set_ylabel("OHC")
axes[0].legend()

bias_mean.plot(ax=axes[1])
axes[1].axhline(0, linewidth=1)
axes[1].set_title("Mean OHC drift: emulator - truth")
axes[1].set_ylabel("OHC bias")

rmse_ts.plot(ax=axes[2])
axes[2].set_title("Spatial OHC RMSE over rollout")
axes[2].set_ylabel("OHC RMSE")
axes[2].set_xlabel("time")

fig.suptitle("First-pass Samudra-style OHC emulator drift")
plt.savefig(figdir / "samudra_first_pass_ohc_drift_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Final-month map diagnostics
# -----------------------------------------------------------------------------

itime = -1

final_truth = truth_ohc.isel(time=itime)
final_pred = pred_ohc.isel(time=itime)
final_error = error_ohc.isel(time=itime)

# Robust colour limits.
truth_abs = float(np.nanpercentile(np.abs(final_truth.where(mask_da)), 99))
err_abs = float(np.nanpercentile(np.abs(final_error.where(mask_da)), 99))

fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(18, 5),
    constrained_layout=True,
)

final_truth.where(mask_da).plot(
    ax=axes[0],
    vmin=-truth_abs,
    vmax=truth_abs,
    cmap="RdBu_r",
    add_colorbar=True,
)
axes[0].set_title("Truth OHC, final month")

final_pred.where(mask_da).plot(
    ax=axes[1],
    vmin=-truth_abs,
    vmax=truth_abs,
    cmap="RdBu_r",
    add_colorbar=True,
)
axes[1].set_title("Emulator OHC, final month")

final_error.where(mask_da).plot(
    ax=axes[2],
    vmin=-err_abs,
    vmax=err_abs,
    cmap="RdBu_r",
    add_colorbar=True,
)
axes[2].set_title("Error: emulator - truth")

plt.savefig(figdir / "samudra_first_pass_ohc_final_month_maps.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Save model and rollout outputs
# -----------------------------------------------------------------------------

checkpoint = {
    "model_state_dict": lightning_model.model.state_dict(),
    "lightning_state_dict": lightning_model.state_dict(),
    "norm_variables": norm_variables,
    "OHC_CH": OHC_CH,
    "HF_CH": HF_CH,
    "STRICT_HF_ONLY": STRICT_HF_ONLY,
    "time_start": time_start,
    "train_end_exclusive": train_end_exclusive,
    "rollout_initial_start": rollout_initial_start,
    "rollout_end_exclusive": rollout_end_exclusive,
    "rollout_months": rollout_months,
    "learning_rate": learning_rate,
    "max_epochs": max_epochs,
}

torch.save(checkpoint, outdir / "samudra_first_pass_ohc_emulator.pt")

truth_ds.to_netcdf(outdir / "rollout_truth_ohc_hf.nc")
pred_ds.to_netcdf(outdir / "rollout_predicted_ohc_with_truth_hf.nc")
drift_summary.to_netcdf(outdir / "rollout_drift_summary.nc")

print(f"Saved checkpoint and rollout outputs to: {outdir}")
print(f"Saved figures to: {figdir}")


## Next iterations

This notebook is deliberately a first pass. The obvious next upgrades are:

1. **Multi-step rollout loss.** Train with a cumulative loss over 3, 6, or 12 recurrent months rather than one-step MSE only.
2. **Tendency prediction.** Predict \(\Delta \mathrm{OHC}\) instead of absolute \(\mathrm{OHC}(t+1)\), which may be better aligned with heat-flux forcing.
3. **More forcing channels.** Add wind stress, freshwater flux, SST / air temperature, or sea-ice fields if available.
4. **Physical constraints.** Add global heat-budget penalties, regional OHC conservation diagnostics, or penalty terms for unrealistic OHC drift.
5. **Architecture upgrade.** Replace the current UNet with ConvNeXt-UNet blocks once the PET sequence workflow is working.
6. **Out-of-distribution test.** Train on one forcing regime and roll forward under a distinct regime to test generalisation.
